# 0.3 RPC BAL RLP Estimation

This notebook estimates raw RLP BAL bytes using JSON-RPC `debug_traceBlockByNumber` with `prestateTracer`, following Toni's `eth-bal-analysis` builder logic.

It does **not** compute calldata from RPC. It reads calldata bytes from the CSV produced by `0.2-calldata-xatu.ipynb`, writes a separate RPC BAL summary CSV, and merges BAL bytes back into the calldata CSV.

## Bandwidth Join

```text
bandwidth_rlp_bytes = xatu_calldata_bytes + rpc_bal_rlp_bytes
```

Each BAL account entry is encoded as:

```text
[address, storage_writes, storage_reads, balance_changes, nonce_changes, code_changes]
```

In [1]:
import importlib
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import sim.rpc_bal as rpc_bal
rpc_bal = importlib.reload(rpc_bal)

BAL_SEMANTICS = rpc_bal.BAL_SEMANTICS
build_rpc_bal_for_block = rpc_bal.build_rpc_bal_for_block

load_dotenv(PROJECT_ROOT / ".env")

RPC_URL = os.environ.get(
    "ETHNODEOPS_RPC",
    "https://erigon.mainnet.rpc.ethnodeops.xyz",
)
ETHNODEOPS_API_KEY = os.environ.get("ETHNODEOPS_API_KEY") or os.environ.get("hoodi_api_key")
if not ETHNODEOPS_API_KEY:
    raise RuntimeError("Missing ETHNODEOPS_API_KEY in .env")

RPC_HEADERS = {"X-API-Key": ETHNODEOPS_API_KEY}
RPC_PROVIDER_LABEL = "ethnodeops_erigon_mainnet"
print("Loaded ETHNODEOPS_API_KEY; using ethnodeops Erigon mainnet RPC")

Loaded ETHNODEOPS_API_KEY; using ethnodeops Erigon mainnet RPC


In [2]:
START_BLOCK = 24_120_001
N_BLOCKS = 50
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))
CALLDATA_CSV = PROJECT_ROOT / "data" / f"xatu_calldata_{min(BLOCKS)}_{max(BLOCKS)}.csv"
SUMMARY_CSV = PROJECT_ROOT / "data" / f"rpc_bal_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"

INCLUDE_READS = True

# True estimates the fuller EIP-7928 block-level BAL payload.
INCLUDE_SYSTEM_CHANGES = True
BAL_SEMANTICS_VERSION = BAL_SEMANTICS

WRITE_CSV = True
WRITE_RLP = False

In [3]:
if not CALLDATA_CSV.exists():
    raise FileNotFoundError(
        f"Missing calldata CSV: {CALLDATA_CSV}. Run notebooks/0.2-calldata-xatu.ipynb first."
    )

calldata = pd.read_csv(CALLDATA_CSV)
missing_blocks = sorted(set(BLOCKS) - set(calldata["block_number"].astype(int)))
if missing_blocks:
    raise RuntimeError(f"Calldata CSV is missing blocks: {missing_blocks}")

calldata_by_block = calldata.set_index("block_number")["calldata_bytes"].astype(int).to_dict()
display(calldata)

,block_number,slot,n_txs_from_payload,n_txs,calldata_bytes,execution_tx_rows,execution_calldata_bytes,calldata_zero_bytes,calldata_nonzero_bytes,calldata_gas,calldata_gas_source,execution_rows_n_input_positive,execution_matches_beacon,cbt_gas_block_size,calldata_gas_minus_cbt_gas_block_size,calldata_gas_matches_cbt_gas_block_size,execution_tx_row_delta,execution_calldata_delta
0,24120001,13350651,344,344,238525,344,238525,144495,94030,2082460,execution_transaction,237,True,1960800,121660,False,0,0
1,24120002,13350652,179,179,44492,179,44492,29852,14640,353648,execution_transaction,117,True,1020300,-666652,False,0,0
2,24120003,13350653,489,489,136567,489,136567,86377,50190,1148548,execution_transaction,377,True,2787300,-1638752,False,0,0
3,24120004,13350654,232,232,70460,232,70460,44058,26402,598664,execution_transaction,166,True,1322400,-723736,False,0,0
4,24120005,13350655,250,250,46996,250,46996,30210,16786,389416,execution_transaction,163,True,1425000,-1035584,False,0,0
5,24120006,13350656,242,242,101974,242,101974,57426,44548,942472,execution_transaction,164,True,1379400,-436928,False,0,0
6,24120007,13350657,69,69,12220,69,12220,6755,5465,114460,execution_transaction,38,True,393300,-278840,False,0,0
7,24120008,13350658,483,483,133308,483,133308,89789,43519,1055460,execution_transaction,298,True,2753100,-1697640,False,0,0
8,24120009,13350659,256,256,91881,256,91881,62112,29769,724752,execution_transaction,165,True,1459200,-734448,False,0,0
9,24120010,13350660,416,416,145880,416,145880,81120,64760,1360640,execution_transaction,298,True,2371200,-1010560,False,0,0


In [4]:
summary_cols = [
    "block_number",
    "rpc_provider",
    "bal_semantics",
    "include_reads",
    "include_system_changes",
    "calldata_source",
    "calldata_bytes",
    "bal_rlp_bytes",
    "bandwidth_rlp_bytes",
    "accounts",
    "storage_write_slots",
    "storage_write_changes",
    "storage_reads",
    "balance_changes",
    "nonce_changes",
    "code_changes",
    "code_bytes",
    "storage_writes_rlp_bytes",
    "storage_reads_rlp_bytes",
    "balance_changes_rlp_bytes",
    "nonce_changes_rlp_bytes",
    "code_changes_rlp_bytes",
    "account_shell_rlp_bytes",
]

rows = []
if SUMMARY_CSV.exists():
    prior = pd.read_csv(SUMMARY_CSV)
    required_cache_cols = {"rpc_provider", "bal_semantics", "include_reads", "include_system_changes"}
    if required_cache_cols.issubset(prior.columns):
        prior = prior[
            (prior["rpc_provider"] == RPC_PROVIDER_LABEL)
            & (prior["bal_semantics"] == BAL_SEMANTICS_VERSION)
            & (prior["include_reads"].astype(bool) == INCLUDE_READS)
            & (prior["include_system_changes"].astype(bool) == INCLUDE_SYSTEM_CHANGES)
        ]
    else:
        prior = prior.iloc[0:0]
    available_summary_cols = [col for col in summary_cols if col in prior.columns]
    rows.extend(prior[available_summary_cols].to_dict("records"))

seen = {int(row["block_number"]) for row in rows}
rlp_outputs = {}

for block_number in BLOCKS:
    if int(block_number) in seen:
        print(f"Skipping block {block_number}; already in {SUMMARY_CSV.name}")
        continue
    print(f"Building RPC BAL for block {block_number}...")
    result = build_rpc_bal_for_block(
        RPC_URL,
        block_number,
        calldata_bytes=calldata_by_block[block_number],
        rpc_headers=RPC_HEADERS,
        include_reads=INCLUDE_READS,
        include_system_changes=INCLUDE_SYSTEM_CHANGES,
    )
    row = result.summary.as_dict()
    row["rpc_provider"] = RPC_PROVIDER_LABEL
    rows.append(row)
    seen.add(int(block_number))
    rlp_outputs[block_number] = result.rlp_bytes
    if WRITE_CSV:
        data_dir = PROJECT_ROOT / "data"
        data_dir.mkdir(exist_ok=True)
        pd.DataFrame(rows).reindex(columns=summary_cols).drop_duplicates("block_number", keep="last").sort_values("block_number").to_csv(SUMMARY_CSV, index=False)

summary = pd.DataFrame(rows).reindex(columns=summary_cols).drop_duplicates("block_number", keep="last").sort_values("block_number")
display(summary)

if WRITE_CSV:
    data_dir = PROJECT_ROOT / "data"
    data_dir.mkdir(exist_ok=True)
    summary.to_csv(SUMMARY_CSV, index=False)
    print(SUMMARY_CSV)

merge_cols = [col for col in summary_cols if col not in {"calldata_bytes", "calldata_source"}]
stale_cols = [col for col in merge_cols if col != "block_number" and col in calldata.columns]
merged = calldata.drop(columns=stale_cols + [col for col in ["bal_bytes", "bandwidth_bytes"] if col in calldata.columns])
merged = merged.merge(summary[merge_cols], on="block_number", how="left", validate="one_to_one")
merged["bal_bytes"] = merged["bal_rlp_bytes"].astype("Int64")
merged["bandwidth_bytes"] = (merged["calldata_bytes"] + merged["bal_bytes"]).astype("Int64")

front = []
for col in merged.columns:
    if col in {"bal_bytes", "bandwidth_bytes"}:
        continue
    front.append(col)
    if col == "calldata_bytes":
        front.extend(["bal_bytes", "bandwidth_bytes"])
merged = merged[front + [col for col in merged.columns if col not in front]]
display(merged)

if WRITE_CSV:
    merged.to_csv(CALLDATA_CSV, index=False)
    print(CALLDATA_CSV)

if WRITE_RLP:
    suffix = "with_reads" if INCLUDE_READS else "without_reads"
    for block_number, payload in rlp_outputs.items():
        data_dir = PROJECT_ROOT / "data"
        data_dir.mkdir(exist_ok=True)
        out = data_dir / f"rpc_bal_{block_number}_{suffix}.rlp"
        out.write_bytes(payload)
        print(out)

Skipping block 24120001; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120002; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120003; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120004; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120005; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120006; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120007; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120008; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120009; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120010; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120011; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120012; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120013; already in rpc_bal_summary_24120001_24120050.csv
Skipping block 24120014; already in rp

,block_number,rpc_provider,bal_semantics,include_reads,include_system_changes,calldata_source,calldata_bytes,bal_rlp_bytes,bandwidth_rlp_bytes,accounts,...,balance_changes,nonce_changes,code_changes,code_bytes,storage_writes_rlp_bytes,storage_reads_rlp_bytes,balance_changes_rlp_bytes,nonce_changes_rlp_bytes,code_changes_rlp_bytes,account_shell_rlp_bytes
0,24120001,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,238525,339327,577852,892,...,962,350,4,1358,233907,66719,11942,2117,1392,23250
1,24120002,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,44492,72309,116801,432,...,474,182,2,46,35712,18712,5521,1036,56,11272
2,24120003,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,136567,186956,323523,1088,...,1299,493,2,1217,85070,53560,15894,2975,1233,28224
3,24120004,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,70460,117839,188299,647,...,655,236,2,3152,55731,33198,7534,1307,3166,16903
4,24120005,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,46996,81624,128620,527,...,644,252,2,1110,37902,20062,7465,1400,1125,13670
5,24120006,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,101974,129883,231857,710,...,759,250,6,20403,59864,20972,8692,1415,20445,18495
6,24120007,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,12220,19240,31460,188,...,232,69,0,0,7149,4168,2681,364,0,4878
7,24120008,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,133308,175656,308964,1060,...,1321,498,10,5120,85373,38718,15839,3023,5206,27497
8,24120009,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,91881,109312,201193,592,...,738,260,3,1219,49780,33141,8392,1420,1239,15340
9,24120010,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,xatu,145880,169659,315539,910,...,1166,422,4,28523,65982,34723,14241,2561,28564,23588


/Users/william/PycharmProjects/eip-7999-research/data/rpc_bal_summary_24120001_24120050.csv


,block_number,slot,n_txs_from_payload,n_txs,calldata_bytes,bal_bytes,bandwidth_bytes,execution_tx_rows,execution_calldata_bytes,calldata_zero_bytes,...,balance_changes,nonce_changes,code_changes,code_bytes,storage_writes_rlp_bytes,storage_reads_rlp_bytes,balance_changes_rlp_bytes,nonce_changes_rlp_bytes,code_changes_rlp_bytes,account_shell_rlp_bytes
0,24120001,13350651,344,344,238525,339327,577852,344,238525,144495,...,962,350,4,1358,233907,66719,11942,2117,1392,23250
1,24120002,13350652,179,179,44492,72309,116801,179,44492,29852,...,474,182,2,46,35712,18712,5521,1036,56,11272
2,24120003,13350653,489,489,136567,186956,323523,489,136567,86377,...,1299,493,2,1217,85070,53560,15894,2975,1233,28224
3,24120004,13350654,232,232,70460,117839,188299,232,70460,44058,...,655,236,2,3152,55731,33198,7534,1307,3166,16903
4,24120005,13350655,250,250,46996,81624,128620,250,46996,30210,...,644,252,2,1110,37902,20062,7465,1400,1125,13670
5,24120006,13350656,242,242,101974,129883,231857,242,101974,57426,...,759,250,6,20403,59864,20972,8692,1415,20445,18495
6,24120007,13350657,69,69,12220,19240,31460,69,12220,6755,...,232,69,0,0,7149,4168,2681,364,0,4878
7,24120008,13350658,483,483,133308,175656,308964,483,133308,89789,...,1321,498,10,5120,85373,38718,15839,3023,5206,27497
8,24120009,13350659,256,256,91881,109312,201193,256,91881,62112,...,738,260,3,1219,49780,33141,8392,1420,1239,15340
9,24120010,13350660,416,416,145880,169659,315539,416,145880,81120,...,1166,422,4,28523,65982,34723,14241,2561,28564,23588


/Users/william/PycharmProjects/eip-7999-research/data/xatu_calldata_24120001_24120050.csv


In [5]:
# Optional local calibration against nerolation/eth-bal-analysis raw RLP samples.
sample_dir = Path("/private/tmp/eth-bal-analysis/bal_raw/rlp")
calibration_rows = []
if sample_dir.exists():
    suffix = "with_reads" if INCLUDE_READS else "without_reads"
    for block_number in BLOCKS:
        sample = sample_dir / f"{block_number}_{suffix}.rlp"
        if sample.exists():
            sample_bytes = sample.stat().st_size
            row = summary[summary["block_number"] == block_number].iloc[0]
            calibration_rows.append({
                "block_number": block_number,
                "rpc_bal_rlp_bytes": int(row["bal_rlp_bytes"]),
                "sample_bal_rlp_bytes": sample_bytes,
                "delta_bytes": int(row["bal_rlp_bytes"]) - sample_bytes,
            })

calibration = pd.DataFrame(calibration_rows)
display(calibration)

""
